In [ ]:
from pathlib import Path
import re
import unicodedata
import csv
import pandas as pd
import numpy as np

# Dependencias necesarias en el entorno:
# %pip install openpyxl pyarrow


In [ ]:
import os
# La ruta de datos se puede fijar desde fuera (pipeline_mensual.py) con la
# variable de entorno EBSA_DATOS. Si no existe, se usa la de siempre.
DATA_DIR = Path(os.environ.get("EBSA_DATOS", r"C:\Users\Home\Documents\Datos_Ebsa"))
# Los archivos originales de la empresa (formato TC2) van en 00_formato_TC2
ORIGEN_DIR = DATA_DIR / "00_formato_TC2"
ORIGEN_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR = DATA_DIR / "01_historico_procesado"
DETAIL_DIR = OUT_DIR / "detalle_mensual"
# Resumen NIU x mes de cada archivo ya procesado: si el archivo no cambió, no se vuelve a leer.
RESUMEN_DIR = OUT_DIR / "resumen_por_archivo"
RESUMEN_DIR.mkdir(parents=True, exist_ok=True)

OUT_DIR.mkdir(parents=True, exist_ok=True)
DETAIL_DIR.mkdir(parents=True, exist_ok=True)


EXTENSIONES_VALIDAS = {".xlsx", ".csv"}

archivos = sorted(
    archivo
    for archivo in ORIGEN_DIR.iterdir()
    if archivo.is_file()
    and archivo.suffix.lower() in EXTENSIONES_VALIDAS
    and not archivo.name.startswith("~$")
)

print(f"Archivos encontrados: {len(archivos)}")

if not archivos:
    raise FileNotFoundError(
        f"No hay archivos .xlsx ni .csv en:\n{ORIGEN_DIR}\n\n"
        "Copia ahí los archivos mensuales de la empresa (formato TC2). "
        "Si en vez de eso copiaste los historico_YYYY.parquet a 01_historico_procesado, "
        "este paso no hace falta: corre el pipeline con --desde 2."
    )

for archivo in archivos:
    print(f"{archivo.suffix.lower():<6} | {archivo.name}")
print(f"Archivos encontrados: {len(archivos)}")

for archivo in archivos[:10]:
    print(archivo.name)


In [ ]:
# ============================================================
# ESQUEMA CANÓNICO
# ============================================================
# Estos son los nombres internos que usaremos SIEMPRE,
# aunque un archivo de otro año tenga una variante del encabezado.

COLUMNAS_ESENCIALES = [
    "NIU",
    "Tipo Factura",
    "Tipo de Lectura",
    "Consumo Usuario (kWh)",
    "Consumo Promedio Semestral (kWh)",
    "Tipo Medidor",
    "Tarifa Aplicada ($/kWh)"
]


# Estas columnas son útiles pero NO son obligatorias.
# Si no existen, el procesamiento continúa.

COLUMNAS_OPCIONALES = [
    "Año de reporte",
    "Mes de reporte",
    "Estrato",
    "ID Factura",
    "Días Facturados",
    "Fecha de Lectura Actual",
    "Fecha de Lectura Anterior",
    "Refacturación por Consumo Usuario - (kWh)",
    "Ciclo",
    "Clase de Servicio",
    "Valor Facturación por Consumo Usuario ($)"
]


# Todas son buscadas en los archivos,
# pero solamente COLUMNAS_ESENCIALES provocan error.

COLUMNAS_OBJETIVO = list(
    dict.fromkeys(
        COLUMNAS_ESENCIALES
        + COLUMNAS_OPCIONALES
    )
)


# ============================================================
# ALIAS DE ENCABEZADOS
# ============================================================
# No hacemos fuzzy matching automático porque puede confundir
# columnas parecidas pero con significado distinto, por ejemplo:
# "Estrato" != necesariamente "Estrato TC1"
# "Tarifa Aplicada ($/kWh)" != necesariamente "CU Tarifa"

ALIASES_COLUMNAS = {
    "NIU": [
        "NIU",
        "N.I.U",
    ],
    "Tipo Factura": [
        "Tipo Factura",
        "Tipo de Factura",
    ],
    "Tipo de Lectura": [
        "Tipo de Lectura",
        "Tipo Lectura",
    ],
    "Consumo Usuario (kWh)": [
        "Consumo Usuario (kWh)",
        "Consumo Usuario kWh",
        "Consumo del Usuario (kWh)",
        "Consumo del Usuario kWh",
    ],
    "Consumo Promedio Semestral (kWh)": [
        "Consumo Promedio Semestral (kWh)",
        "Consumo Promedio Semestral kWh",
        "Promedio Semestral Consumo (kWh)",
    ],
    "Tipo Medidor": [
        "Tipo Medidor",
        "Tipo de Medidor",
    ],
    "Tarifa Aplicada ($/kWh)": [
        "Tarifa Aplicada ($/kWh)",
        "Tarifa Aplicada (COP/kWh)",
        "Tarifa Aplicada kWh",
        "Tarifa Aplicada",
    ],
    "Valor Facturación por Consumo Usuario ($)": [
        "Valor Facturación por Consumo Usuario ($)",
        "Valor Facturacion por Consumo Usuario ($)",
        "Valor Facturación por Consumo Usuario",
        "Valor Facturacion por Consumo Usuario",
        "Valor Facturación Consumo Usuario ($)",
    ],
    "Año de reporte": [
        "Año de reporte",
        "Año reporte",
        "Ano de reporte",
        "Ano reporte",
        "Anio de reporte",
        "Anio reporte",
    ],
    "Mes de reporte": [
        "Mes de reporte",
        "Mes reporte",
    ],
    "Estrato": [
        "Estrato",
        "Estrato Socioeconómico",
        "Estrato Socioeconomico",
    ],
    "ID Factura": [
        "ID Factura",
        "Id Factura",
        "ID de Factura",
        "Identificador Factura",
    ],
    "Días Facturados": [
        "Días Facturados",
        "Dias Facturados",
        "Días de Facturación",
        "Dias de Facturacion",
    ],
    "Fecha de Lectura Actual": [
        "Fecha de Lectura Actual",
        "Fecha Lectura Actual",
    ],
    "Fecha de Lectura Anterior": [
        "Fecha de Lectura Anterior",
        "Fecha Lectura Anterior",
    ],
    "Refacturación por Consumo Usuario - (kWh)": [
        "Refacturación por Consumo Usuario - (kWh)",
        "Refacturacion por Consumo Usuario - (kWh)",
        "Refacturación por Consumo Usuario (kWh)",
        "Refacturacion por Consumo Usuario (kWh)",
    ],
    "Ciclo": [
        "Ciclo",
        "Ciclo Facturación",
        "Ciclo Facturacion",
    ],
    "Clase de Servicio": [
        "Clase de Servicio",
        "Clase Servicio",
    ],
}


def normalizar_nombre_columna(nombre):
    """
    Normaliza un encabezado solo para compararlo:
    - elimina tildes
    - pasa a minúsculas
    - elimina saltos de línea
    - homogeneiza signos/puntuación y espacios
    """
    texto = str(nombre).replace("\n", " ").replace("\r", " ").strip()
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    texto = texto.lower()
    texto = re.sub(r"[^a-z0-9]+", " ", texto)
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto


def resolver_columnas(encabezados):
    """
    Busca en un archivo las columnas equivalentes al esquema canónico.

    Retorna:
      - mapeo: {nombre_real_en_excel: nombre_canonico}
      - faltantes: columnas canónicas no encontradas
      - ambiguas: columnas donde más de un encabezado podría coincidir
    """
    encabezados = list(encabezados)

    por_normalizado = {}
    for original in encabezados:
        clave = normalizar_nombre_columna(original)
        por_normalizado.setdefault(clave, []).append(original)

    mapeo = {}
    ambiguas = {}

    for canonica in COLUMNAS_OBJETIVO:
        alias = [canonica] + ALIASES_COLUMNAS.get(canonica, [])
        claves_validas = {
            normalizar_nombre_columna(x)
            for x in alias
        }

        candidatos = []
        for clave in claves_validas:
            candidatos.extend(por_normalizado.get(clave, []))

        # Eliminar duplicados conservando orden
        candidatos = list(dict.fromkeys(candidatos))

        if not candidatos:
            continue

        # Si existe exactamente el nombre canónico, tiene prioridad.
        if canonica in candidatos:
            elegida = canonica
        else:
            elegida = candidatos[0]

        mapeo[elegida] = canonica

        if len(candidatos) > 1:
            ambiguas[canonica] = candidatos

    encontradas = set(mapeo.values())
    faltantes = [
        col for col in COLUMNAS_OBJETIVO
        if col not in encontradas
    ]

    return mapeo, faltantes, ambiguas

# ============================================================
# DETECCIÓN AUTOMÁTICA DE LA HOJA QUE CONTIENE LOS DATOS TC2
# ============================================================

def detectar_hoja_datos_excel(ruta, mostrar=False):
    """
    Revisa todas las hojas de un Excel y selecciona
    automáticamente la que mejor coincide con el esquema TC2.

    La selección se basa en la cantidad de columnas canónicas
    reconocidas por resolver_columnas().
    """

    excel = pd.ExcelFile(
        ruta,
        engine="openpyxl"
    )

    candidatos = []

    for hoja in excel.sheet_names:

        try:

            encabezados = pd.read_excel(
                excel,
                sheet_name=hoja,
                nrows=0
            ).columns.tolist()

            mapeo, faltantes, ambiguas = (
                resolver_columnas(encabezados)
            )

            columnas_reconocidas = len(mapeo)

            # Variables especialmente importantes
            columnas_clave = {
                "NIU",
                "Consumo Usuario (kWh)",
                "Tipo de Lectura",
                "Año de reporte",
                "Mes de reporte"
            }

            reconocidas = set(
                mapeo.values()
            )

            claves_encontradas = len(
                columnas_clave.intersection(
                    reconocidas
                )
            )

            candidatos.append({
                "hoja": hoja,
                "columnas_totales":
                    len(encabezados),
                "columnas_reconocidas":
                    columnas_reconocidas,
                "claves_encontradas":
                    claves_encontradas
            })

        except Exception as e:

            candidatos.append({
                "hoja": hoja,
                "columnas_totales": 0,
                "columnas_reconocidas": 0,
                "claves_encontradas": 0
            })


    if not candidatos:

        raise ValueError(
            f"No se encontraron hojas en {ruta.name}"
        )


    candidatos_df = pd.DataFrame(
        candidatos
    )


    # --------------------------------------------------------
    # Priorizamos:
    # 1. Mayor cantidad de variables clave
    # 2. Mayor cantidad de columnas reconocidas
    # 3. Mayor cantidad total de columnas
    # --------------------------------------------------------

    mejor = (
        candidatos_df
        .sort_values(
            [
                "claves_encontradas",
                "columnas_reconocidas",
                "columnas_totales"
            ],
            ascending=False
        )
        .iloc[0]
    )


    if mostrar:

        print(
            f"\nArchivo: {ruta.name}"
        )

        display(candidatos_df)

        print(
            f"Hoja seleccionada: "
            f"{mejor['hoja']}"
        )


    # Seguridad mínima:
    # una hoja TC2 debería tener NIU y varias
    # columnas reconocidas.
    if (
        mejor["columnas_reconocidas"] < 3
        and
        mejor["claves_encontradas"] == 0
    ):

        raise ValueError(
            f"No se pudo identificar una hoja TC2 "
            f"válida en {ruta.name}"
        )


    return mejor["hoja"]

def extraer_periodo_nombre(ruta):
    """
    Fallback para recuperar año/mes desde nombres como:
      formato_tc2_20251.xlsx
      formato_tc2_202510.xlsx
      formato_tc2_20251 (1).xlsx
    """
    stem = Path(ruta).stem

    match = re.search(
        r"(20\d{2})(1[0-2]|0?[1-9])(?!\d)",
        stem
    )

    if not match:
        return None, None

    return int(match.group(1)), int(match.group(2))


# ============================================================
# AUDITORÍA ROBUSTA DE ESQUEMAS
# EXCEL + CSV
# ============================================================

def auditar_esquemas(archivos):
    
    filas = []
    
    for ruta in archivos:
        
        print(f"Auditando: {ruta.name}")
        
        try:
            
            # ----------------------------------
            # Obtener encabezados
            # ----------------------------------
            
            encabezados = leer_encabezados(ruta)
            
            
            # ----------------------------------
            # Resolver nombres canónicos
            # ----------------------------------
            
            mapeo, faltantes, ambiguas = (
                resolver_columnas(encabezados)
            )
            
            
            # ----------------------------------
            # Periodo desde nombre archivo
            # ----------------------------------
            
            año, mes = extraer_periodo_nombre(ruta)
            
            
            # ----------------------------------
            # Información del CSV
            # ----------------------------------
            
            if ruta.suffix.lower() == ".csv":
                
                config_csv = (
                    detectar_configuracion_csv(ruta)
                )
                
                separador = repr(config_csv["sep"])
                encoding = config_csv["encoding"]
                
            else:
                
                separador = ""
                encoding = ""
            
            
            filas.append({
                
                "archivo": ruta.name,
                "formato": ruta.suffix.lower(),
                "estado": "OK",
                
                "columnas_archivo":
                    len(encabezados),
                
                "columnas_reconocidas":
                    len(mapeo),
                
                "faltantes":
                    ", ".join(faltantes),
                
                "ambiguas":
                    str(ambiguas)
                    if ambiguas
                    else "",
                
                "año_nombre": año,
                "mes_nombre": mes,
                
                "separador_csv": separador,
                "encoding_csv": encoding,
                
                "error": ""
            })
        
        
        except Exception as e:
            
            filas.append({
                
                "archivo": ruta.name,
                "formato": ruta.suffix.lower(),
                "estado": "ERROR",
                
                "columnas_archivo": None,
                "columnas_reconocidas": None,
                
                "faltantes": "",
                "ambiguas": "",
                
                "año_nombre": None,
                "mes_nombre": None,
                
                "separador_csv": "",
                "encoding_csv": "",
                
                "error":
                    f"{type(e).__name__}: {str(e)}"
            })
    
    
    return pd.DataFrame(filas)


In [ ]:
# ============================================================
# LEER DATOS
# XLSX / CSV
# ============================================================

def leer_datos(ruta, columnas_reales):

    extension = ruta.suffix.lower()


    # ========================================================
    # EXCEL
    # ========================================================

    if extension == ".xlsx":

        hoja = detectar_hoja_datos_excel(
            ruta
        )

        return pd.read_excel(
            ruta,
            sheet_name=hoja,
            usecols=columnas_reales,
            engine="openpyxl"
        )


    # ========================================================
    # CSV
    # ========================================================

    elif extension == ".csv":

        config = detectar_configuracion_csv(
            ruta
        )

        return pd.read_csv(
            ruta,
            usecols=columnas_reales,
            sep=config["sep"],
            encoding=config["encoding"],
            low_memory=False
        )


    else:

        raise ValueError(
            f"Formato no soportado: {extension}"
        )

In [ ]:
# ============================================================
# PROCESAR ARCHIVO
# XLSX / CSV
# ============================================================

def procesar_archivo(
    ruta,
    mostrar_detalle=False
):

    print(
        f"Procesando: {ruta.name} "
        f"[{ruta.suffix.lower()}]"
    )

    # ========================================================
    # 1. OBTENER AÑO Y MES DESDE EL NOMBRE DEL ARCHIVO
    # ========================================================

    año_nombre, mes_nombre = (
        extraer_periodo_nombre(ruta)
    )

    if año_nombre is None or mes_nombre is None:

        raise ValueError(
            f"No fue posible obtener año/mes "
            f"desde el nombre: {ruta.name}"
        )


    # ========================================================
    # 2. LEER ENCABEZADOS
    # ========================================================

    encabezados = leer_encabezados(ruta)


    # ========================================================
    # 3. IDENTIFICAR COLUMNAS
    # ========================================================

    mapeo, faltantes, ambiguas = (
        resolver_columnas(encabezados)
    )


    # ========================================================
    # 4. VALIDAR SOLO COLUMNAS REALMENTE ESENCIALES
    # ========================================================

    faltantes_esenciales = [

        columna

        for columna in COLUMNAS_ESENCIALES

        if columna in faltantes
    ]


    if faltantes_esenciales:

        raise ValueError(

            f"{ruta.name} no contiene "
            f"columnas esenciales: "
            f"{faltantes_esenciales}"
        )


    # ========================================================
    # 5. LEER SOLAMENTE COLUMNAS RECONOCIDAS
    # ========================================================

    columnas_reales = list(
        mapeo.keys()
    )

    df = leer_datos(
        ruta,
        columnas_reales
    )


    # ========================================================
    # 6. RENOMBRAR AL ESQUEMA CANÓNICO
    # ========================================================

    df = df.rename(
        columns=mapeo
    )


    # ========================================================
    # 7. NIU
    # ========================================================

    df["NIU"] = (
        df["NIU"]
        .astype("string")
        .str.strip()
    )


    # Quitar NIU realmente vacíos
    df = df[
        df["NIU"].notna()
        &
        (df["NIU"] != "")
    ].copy()


    # ========================================================
    # 8. GUARDAR AÑO/MES ORIGINALES PARA AUDITORÍA
    # ========================================================
    #
    # No los utilizaremos para construir el periodo,
    # pero conservarlos sirve para descubrir errores
    # dentro de los archivos originales.
    # ========================================================

    if "Año de reporte" in df.columns:

        df["Año_reporte_original"] = (
            pd.to_numeric(
                df["Año de reporte"],
                errors="coerce"
            )
        )


    if "Mes de reporte" in df.columns:

        df["Mes_reporte_original"] = (
            pd.to_numeric(
                df["Mes de reporte"],
                errors="coerce"
            )
        )


    # ========================================================
    # 9. EL NOMBRE DEL ARCHIVO ES LA FUENTE OFICIAL
    # ========================================================
    #
    # Ejemplo:
    #
    # formato_tc2_20255.xlsx
    #
    # SIEMPRE:
    # Año = 2025
    # Mes = 5
    #
    # aunque dentro del archivo existan valores incorrectos.
    # ========================================================

    df["Año de reporte"] = año_nombre

    df["Mes de reporte"] = mes_nombre


    # ========================================================
    # 10. CREAR PERIODO
    # ========================================================

    df["periodo"] = pd.Timestamp(
        year=int(año_nombre),
        month=int(mes_nombre),
        day=1
    )


    # ========================================================
    # 11. CONVERSIÓN DE VARIABLES NUMÉRICAS
    # ========================================================

    columnas_numericas = [

        "Consumo Usuario (kWh)",

        "Consumo Promedio Semestral (kWh)",

        "Tarifa Aplicada ($/kWh)",

        "Días Facturados",

        "Refacturación por Consumo Usuario - (kWh)",

        "Ciclo",

        "Estrato"
    ]


    for columna in columnas_numericas:

        if columna in df.columns:

            df[columna] = pd.to_numeric(
                df[columna],
                errors="coerce"
            )


    # ========================================================
    # 12. FECHAS DE LECTURA
    # ========================================================

    columnas_fecha = [

        "Fecha de Lectura Actual",

        "Fecha de Lectura Anterior"
    ]


    for columna in columnas_fecha:

        if columna in df.columns:

            df[columna] = pd.to_datetime(
                df[columna],
                errors="coerce",
                dayfirst=True
            )


    # ========================================================
    # 13. ESTRATO ES OPCIONAL
    # ========================================================
    #
    # Si el archivo no tiene Estrato,
    # creamos la columna vacía.
    #
    # Así todos los Parquet mantienen
    # una estructura compatible.
    # ========================================================

    if "Estrato" not in df.columns:

        df["Estrato"] = pd.NA


    # ========================================================
    # 14. VARIABLES DE CONTROL
    # ========================================================

    df["archivo_origen"] = ruta.name

    df["formato_origen"] = (
        ruta.suffix.lower()
    )


    # ========================================================
    # 15. DETECTAR INCONSISTENCIAS AÑO/MES
    # ========================================================

    if "Año_reporte_original" in df.columns:

        df["error_año_original"] = (

            df["Año_reporte_original"].notna()

            &

            (
                df["Año_reporte_original"]
                != año_nombre
            )
        )


    if "Mes_reporte_original" in df.columns:

        df["error_mes_original"] = (

            df["Mes_reporte_original"].notna()

            &

            (
                df["Mes_reporte_original"]
                != mes_nombre
            )
        )


    # ========================================================
    # 15b. COLUMNAS DE TEXTO: TODO A TEXTO
    # ========================================================
    # En algunos archivos una columna de texto trae celdas
    # numéricas (p. ej. Clase de Servicio con un 0). Parquet no
    # acepta una columna con texto y números mezclados, así que
    # todo lo que no sea numérico ni fecha se deja como texto.
    # ========================================================

    for columna in df.columns:

        if df[columna].dtype == object:

            df[columna] = (
                df[columna]
                .astype("string")
                .str.strip()
            )


    # ========================================================
    # 16. INFORMACIÓN OPCIONAL
    # ========================================================

    if mostrar_detalle:

        print(
            f"    Filas: {len(df):,}"
        )

        print(
            f"    NIU únicos: "
            f"{df['NIU'].nunique():,}"
        )

        print(
            f"    Periodo asignado: "
            f"{año_nombre}-{mes_nombre:02d}"
        )

        print(
            f"    Columnas: "
            f"{len(df.columns)}"
        )


    return df

In [ ]:
# ============================================================
# DETECCIÓN AUTOMÁTICA DE CSV
# ============================================================

def detectar_configuracion_csv(ruta):
    
    encodings = [
        "utf-8-sig",
        "utf-8",
        "cp1252",
        "latin1"
    ]
    
    ultimo_error = None
    
    for encoding in encodings:
        
        try:
            
            with open(
                ruta,
                "r",
                encoding=encoding,
                errors="strict"
            ) as f:
                
                muestra = f.read(10000)
            
            # Intentar detectar separador
            try:
                
                dialecto = csv.Sniffer().sniff(
                    muestra,
                    delimiters=",;|\t"
                )
                
                separador = dialecto.delimiter
                
            except csv.Error:
                # El más común en Colombia
                separador = ";"
            
            return {
                "encoding": encoding,
                "sep": separador
            }
        
        except UnicodeDecodeError as e:
            ultimo_error = e
    
    
    raise ValueError(
        f"No se pudo determinar la codificación de {ruta.name}. "
        f"Último error: {ultimo_error}"
    )

In [ ]:
# ============================================================
# LEER ENCABEZADOS
# XLSX: DETECTA AUTOMÁTICAMENTE LA HOJA TC2
# CSV: DETECTA SEPARADOR Y ENCODING
# ============================================================

def leer_encabezados(ruta):

    extension = ruta.suffix.lower()


    # ========================================================
    # EXCEL
    # ========================================================

    if extension == ".xlsx":

        hoja = detectar_hoja_datos_excel(
            ruta
        )

        df = pd.read_excel(
            ruta,
            sheet_name=hoja,
            nrows=0,
            engine="openpyxl"
        )

        return df.columns.tolist()


    # ========================================================
    # CSV
    # ========================================================

    elif extension == ".csv":

        config = detectar_configuracion_csv(
            ruta
        )

        df = pd.read_csv(
            ruta,
            nrows=0,
            sep=config["sep"],
            encoding=config["encoding"]
        )

        return df.columns.tolist()


    else:

        raise ValueError(
            f"Formato no soportado: {extension}"
        )

In [ ]:
# ============================================================
# AUDITORÍA DE ESQUEMAS ANTES DE CARGAR MILLONES DE REGISTROS
# ============================================================

auditoria = auditar_esquemas(archivos)

display(auditoria)

# Archivos que requieren atención:
display(
    auditoria[
        (auditoria["faltantes"] != "")
        | (auditoria["ambiguas"] != "")
    ]
)


In [ ]:
# ============================================================
# PROCESAMIENTO DE TODO EL HISTÓRICO
# ============================================================
# Objetivo:
#
# 1. Procesar cada archivo XLSX / CSV.
# 2. Guardar el DETALLE mensual en Parquet.
# 3. Crear un resumen NIU-periodo.
# 4. CONSERVAR variables de consumo para análisis/modelado.
# 5. No detener todo el proceso si un archivo presenta error.
#
# IMPORTANTE:
# consumo_kwh_raw todavía NO representa necesariamente el
# consumo mensual definitivo porque pueden existir:
# - múltiples facturas NIU-mes
# - refacturaciones
# - lecturas trimestrales
# ============================================================

import pandas as pd


# ============================================================
# CREAR CARPETA DE DETALLE
# ============================================================

DETAIL_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# CONTENEDORES
# ============================================================

resumenes = []

errores_procesamiento = []
archivos_procesados = []      # leídos en esta corrida
archivos_reutilizados = []    # ya estaban procesados (caché)


# ============================================================
# RECORRER TODOS LOS ARCHIVOS
# ============================================================

for numero, archivo in enumerate(
    archivos,
    start=1
):

    print(
        f"\n[{numero}/{len(archivos)}] "
        f"Procesando: {archivo.name}"
    )

    try:

        # ====================================================
        # 0. ¿YA SE PROCESÓ ESTE ARCHIVO? (caché por archivo)
        # ====================================================
        # Si el resumen de este archivo existe y es posterior al
        # archivo mismo, se reutiliza: la corrida mensual solo
        # lee de verdad el archivo nuevo.
        # ====================================================

        formato = archivo.suffix.lower().replace(".", "")
        ruta_resumen_archivo = RESUMEN_DIR / f"{archivo.stem}_{formato}.parquet"

        if (
            ruta_resumen_archivo.exists()
            and ruta_resumen_archivo.stat().st_mtime >= archivo.stat().st_mtime
        ):
            resumen = pd.read_parquet(ruta_resumen_archivo, engine="pyarrow")
            resumen["NIU"] = resumen["NIU"].astype("string").str.strip()
            resumenes.append(resumen)
            archivos_reutilizados.append(archivo.name)
            print(f"    ya procesado antes: se reutiliza ({len(resumen):,} filas NIU-mes)")
            del resumen
            continue

        # ====================================================
        # 1. PROCESAR ARCHIVO
        # ====================================================

        df = procesar_archivo(
            archivo
        )


        # ====================================================
        # 2. VALIDACIONES BÁSICAS
        # ====================================================

        if df.empty:

            raise ValueError(
                "El archivo fue procesado "
                "pero quedó vacío."
            )


        columnas_obligatorias = [
            "NIU",
            "periodo"
        ]


        faltantes_obligatorias = [
            columna
            for columna in columnas_obligatorias
            if columna not in df.columns
        ]


        if faltantes_obligatorias:

            raise ValueError(
                "Faltan columnas necesarias "
                "después del procesamiento: "
                f"{faltantes_obligatorias}"
            )


        # ====================================================
        # 3. INFORMACIÓN DEL ARCHIVO
        # ====================================================

        print(
            f"    Filas: "
            f"{len(df):,}"
        )

        print(
            f"    Columnas: "
            f"{len(df.columns)}"
        )

        print(
            f"    NIU únicos: "
            f"{df['NIU'].nunique():,}"
        )


        if df["periodo"].notna().any():

            print(
                "    Periodo: "
                f"{df['periodo'].min():%Y-%m}"
                " → "
                f"{df['periodo'].max():%Y-%m}"
            )


        # ====================================================
        # 4. GUARDAR DETALLE MENSUAL
        # ====================================================
        #
        # Aquí se conserva la información original procesada:
        # consumo, facturas, tarifas, lecturas, etc.
        # ====================================================

        formato = (
            archivo
            .suffix
            .lower()
            .replace(".", "")
        )


        salida_detalle = (
            DETAIL_DIR
            / f"{archivo.stem}_{formato}.parquet"
        )


        df.to_parquet(
            salida_detalle,
            index=False,
            engine="pyarrow"
        )


        print(
            f"    Parquet detalle: "
            f"{salida_detalle.name}"
        )


        # ====================================================
        # 5. DEFINIR AGREGACIONES NIU - PERIODO
        # ====================================================

        agregaciones = {

            "cantidad_registros": (
                "NIU",
                "size"
            )
        }


        # ====================================================
        # CONSUMO USUARIO
        # ====================================================
        #
        # Se denomina RAW porque todavía debemos validar
        # refacturaciones y múltiples facturas.
        # ====================================================

        if "Consumo Usuario (kWh)" in df.columns:

            agregaciones[
                "consumo_kwh_raw"
            ] = (
                "Consumo Usuario (kWh)",
                lambda x: x.sum(
                    min_count=1
                )
            )


            agregaciones[
                "registros_consumo"
            ] = (
                "Consumo Usuario (kWh)",
                "count"
            )


            agregaciones[
                "consumo_kwh_min"
            ] = (
                "Consumo Usuario (kWh)",
                "min"
            )


            agregaciones[
                "consumo_kwh_max"
            ] = (
                "Consumo Usuario (kWh)",
                "max"
            )


            agregaciones[
                "consumo_kwh_mediana"
            ] = (
                "Consumo Usuario (kWh)",
                "median"
            )


        # ====================================================
        # CONSUMO PROMEDIO SEMESTRAL
        # ====================================================

        if (
            "Consumo Promedio Semestral (kWh)"
            in df.columns
        ):

            agregaciones[
                "consumo_promedio_semestral_kwh"
            ] = (
                "Consumo Promedio Semestral (kWh)",
                "median"
            )


        # ====================================================
        # TARIFA
        # ====================================================

        if (
            "Tarifa Aplicada ($/kWh)"
            in df.columns
        ):

            agregaciones[
                "tarifa_aplicada_kwh"
            ] = (
                "Tarifa Aplicada ($/kWh)",
                "median"
            )


        # ====================================================
        # VALOR FACTURADO POR CONSUMO (columna Q del TC2, $)
        # Se suma por NIU-mes igual que el consumo. Es el
        # valor real facturado, no un cálculo nuestro.
        # ====================================================

        if (
            "Valor Facturación por Consumo Usuario ($)"
            in df.columns
        ):

            agregaciones[
                "valor_facturado_consumo"
            ] = (
                "Valor Facturación por Consumo Usuario ($)",
                lambda x: x.sum(
                    min_count=1
                )
            )


        # ====================================================
        # REFACTURACIÓN DE CONSUMO
        # ====================================================

        if (
            "Refacturación por Consumo Usuario - (kWh)"
            in df.columns
        ):

            agregaciones[
                "refacturacion_consumo_kwh"
            ] = (
                "Refacturación por Consumo Usuario - (kWh)",
                lambda x: x.sum(
                    min_count=1
                )
            )


        # ====================================================
        # DÍAS FACTURADOS
        # ====================================================

        if "Días Facturados" in df.columns:

            agregaciones[
                "dias_facturados_max"
            ] = (
                "Días Facturados",
                "max"
            )


            agregaciones[
                "dias_facturados_mediana"
            ] = (
                "Días Facturados",
                "median"
            )


            agregaciones[
                "dias_facturados_min"
            ] = (
                "Días Facturados",
                "min"
            )


        # ====================================================
        # FECHAS DE LECTURA
        # ====================================================

        if (
            "Fecha de Lectura Actual"
            in df.columns
        ):

            agregaciones[
                "fecha_lectura_actual"
            ] = (
                "Fecha de Lectura Actual",
                "max"
            )


        if (
            "Fecha de Lectura Anterior"
            in df.columns
        ):

            agregaciones[
                "fecha_lectura_anterior"
            ] = (
                "Fecha de Lectura Anterior",
                "min"
            )


        # ====================================================
        # CICLO
        # ====================================================

        if "Ciclo" in df.columns:

            agregaciones[
                "ciclo"
            ] = (
                "Ciclo",
                "first"
            )


            agregaciones[
                "ciclos_diferentes"
            ] = (
                "Ciclo",
                "nunique"
            )


        # ====================================================
        # TIPO DE LECTURA
        # ====================================================

        if "Tipo de Lectura" in df.columns:

            agregaciones[
                "tipo_lectura"
            ] = (
                "Tipo de Lectura",
                "first"
            )


            agregaciones[
                "tipos_lectura_diferentes"
            ] = (
                "Tipo de Lectura",
                "nunique"
            )


        # ====================================================
        # TIPO FACTURA
        # ====================================================

        if "Tipo Factura" in df.columns:

            agregaciones[
                "tipo_factura"
            ] = (
                "Tipo Factura",
                "first"
            )


            agregaciones[
                "tipos_factura_diferentes"
            ] = (
                "Tipo Factura",
                "nunique"
            )


        # ====================================================
        # TIPO MEDIDOR
        # ====================================================

        if "Tipo Medidor" in df.columns:

            agregaciones[
                "tipo_medidor"
            ] = (
                "Tipo Medidor",
                "first"
            )


            agregaciones[
                "tipos_medidor_diferentes"
            ] = (
                "Tipo Medidor",
                "nunique"
            )


        # ====================================================
        # ESTRATO
        # ====================================================

        if "Estrato" in df.columns:

            agregaciones[
                "estrato"
            ] = (
                "Estrato",
                "first"
            )


        # ====================================================
        # CLASE DE SERVICIO
        # ====================================================

        if (
            "Clase de Servicio"
            in df.columns
        ):

            agregaciones[
                "clase_servicio"
            ] = (
                "Clase de Servicio",
                "first"
            )


        # ====================================================
        # 6. CREAR RESUMEN NIU - PERIODO
        # ====================================================

        resumen = (
            df
            .dropna(
                subset=[
                    "NIU",
                    "periodo"
                ]
            )
            .groupby(
                [
                    "NIU",
                    "periodo"
                ],
                as_index=False
            )
            .agg(
                **agregaciones
            )
        )


        # ====================================================
        # 7. VARIABLES DE CONTROL
        # ====================================================

        # Más de un registro para el mismo NIU-mes.
        resumen[
            "tiene_multiples_registros"
        ] = (
            resumen[
                "cantidad_registros"
            ] > 1
        )


        # Indicio de lectura aproximadamente trimestral.
        if (
            "dias_facturados_max"
            in resumen.columns
        ):

            resumen[
                "lectura_aprox_trimestral"
            ] = (
                resumen[
                    "dias_facturados_max"
                ]
                .between(
                    75,
                    105
                )
            )


        # ====================================================
        # 8. INFORMACIÓN DE ORIGEN
        # ====================================================

        resumen[
            "archivo_origen"
        ] = archivo.name

        resumen[
            "formato_origen"
        ] = archivo.suffix.lower()


        # ====================================================
        # 9. GUARDAR RESUMEN EN LISTA (y en disco, para la caché)
        # ====================================================

        resumen.to_parquet(
            ruta_resumen_archivo,
            index=False,
            engine="pyarrow"
        )

        resumenes.append(
            resumen
        )
        archivos_procesados.append(archivo.name)


        print(
            f"    Resumen NIU-periodo: "
            f"{len(resumen):,} filas"
        )


        # Mostrar si consumo quedó incluido
        if (
            "consumo_kwh_raw"
            in resumen.columns
        ):

            print(
                "    ✓ Consumo incluido "
                "en resumen"
            )

        else:

            print(
                "    ⚠ Consumo NO disponible "
                "en este archivo"
            )


        print("    OK")


        # ====================================================
        # 10. LIBERAR MEMORIA
        # ====================================================

        del df
        del resumen


    # ========================================================
    # SI EL ARCHIVO FALLA
    # ========================================================

    except Exception as e:

        print(
            f"    ERROR: "
            f"{type(e).__name__}: "
            f"{e}"
        )


        errores_procesamiento.append({

            "archivo":
                archivo.name,

            "formato":
                archivo.suffix.lower(),

            "tipo_error":
                type(e).__name__,

            "error":
                str(e)
        })


# ============================================================
# RESUMEN FINAL
# ============================================================

print("\n")
print("=" * 60)
print("PROCESAMIENTO TERMINADO")
print("=" * 60)


print(
    f"Archivos encontrados: "
    f"{len(archivos)}"
)


print(
    f"Archivos leídos en esta corrida: "
    f"{len(archivos_procesados)}"
)

print(
    f"Archivos reutilizados (ya procesados): "
    f"{len(archivos_reutilizados)}"
)


print(
    f"Archivos con error: "
    f"{len(errores_procesamiento)}"
)

In [ ]:
# ============================================================
# ANUALIZAR: INCORPORAR CADA MES A SU historico_YYYY.parquet
# ============================================================
# Por cada año que aparece en los archivos leídos:
#   - si historico_YYYY.parquet no existe, se crea con esos meses;
#   - si existe, se conservan sus meses que NO vienen en esta
#     corrida y se agregan los nuevos. Un mes que ya estaba y
#     vuelve a llegar (archivo corregido por la empresa) se
#     REEMPLAZA por el nuevo, y se avisa.
# Los archivos de otros años no se tocan.
# ============================================================

if errores_procesamiento:
    print("⚠ Archivos con error (no se incorporan):")
    for e in errores_procesamiento:
        print(f"   • {e['archivo']}: {e['tipo_error']}: {e['error'][:200]}")

if not resumenes:
    raise ValueError("Ningún archivo pudo procesarse: revisa los errores de arriba.")

historico_nuevo = pd.concat(resumenes, ignore_index=True)
historico_nuevo["NIU"] = historico_nuevo["NIU"].astype("string").str.strip()
historico_nuevo["periodo"] = (
    pd.to_datetime(historico_nuevo["periodo"], errors="coerce").dt.to_period("M").dt.to_timestamp()
)
historico_nuevo = historico_nuevo[historico_nuevo["periodo"].notna()]
del resumenes

# Solo los archivos leídos EN ESTA CORRIDA obligan a reescribir su año; los reutilizados
# ya están dentro del histórico. Si es la primera corrida, todos son nuevos.
periodos_por_archivo = historico_nuevo.groupby("archivo_origen")["periodo"].agg(lambda s: sorted(s.unique()))
periodos_nuevos = sorted({
    p for a in archivos_procesados for p in periodos_por_archivo.get(a, [])
})

print("INCORPORACIÓN AL HISTÓRICO ANUAL")
print("-" * 70)
if not periodos_nuevos:
    print("No hay archivos nuevos: el histórico ya contiene todos los meses de 00_formato_TC2.")
else:
    print("Meses que entran o se reemplazan:", ", ".join(f"{p:%Y-%m}" for p in periodos_nuevos))

for anio in sorted({p.year for p in periodos_nuevos}):
    ruta_anual = OUT_DIR / f"historico_{anio}.parquet"
    meses_anio = [p for p in periodos_nuevos if p.year == anio]
    nuevo = historico_nuevo[historico_nuevo["periodo"].isin(meses_anio)].copy()

    if ruta_anual.exists():
        viejo = pd.read_parquet(ruta_anual, engine="pyarrow")
        viejo["NIU"] = viejo["NIU"].astype("string").str.strip()
        viejo["periodo"] = pd.to_datetime(viejo["periodo"], errors="coerce").dt.to_period("M").dt.to_timestamp()
        ya_estaban = sorted(set(viejo["periodo"].unique()) & set(meses_anio))
        if ya_estaban:
            print(f"  ⚠ {ruta_anual.name}: los meses "
                  + ", ".join(f"{p:%Y-%m}" for p in ya_estaban)
                  + " ya existían y se REEMPLAZAN por el archivo nuevo.")
        conservar = viejo[~viejo["periodo"].isin(meses_anio)]
        combinado = pd.concat([conservar, nuevo], ignore_index=True)
        accion = f"actualizado: +{len(meses_anio)} mes(es), conserva {conservar['periodo'].nunique()} anteriores"
        del viejo, conservar
    else:
        combinado = nuevo
        accion = f"creado con {len(meses_anio)} mes(es)"

    combinado = combinado.sort_values(["periodo", "NIU"]).reset_index(drop=True)
    combinado.to_parquet(ruta_anual, index=False, engine="pyarrow")
    print(f"  ✓ {ruta_anual.name} {accion}  "
          f"({combinado['periodo'].min():%Y-%m} -> {combinado['periodo'].max():%Y-%m}, "
          f"{len(combinado):,} filas, {combinado['NIU'].nunique():,} NIU)")
    del combinado, nuevo

del historico_nuevo


In [ ]:
# ============================================================
# COBERTURA DEL HISTÓRICO COMPLETO (todos los historico_YYYY.parquet)
# ============================================================
import pyarrow.parquet as pq

archivos_anuales = sorted(OUT_DIR.glob("historico_*.parquet"))
partes = []
for ruta in archivos_anuales:
    cols = [c for c in ["NIU", "periodo", "consumo_kwh_raw"] if c in pq.ParquetFile(ruta).schema_arrow.names]
    partes.append(pd.read_parquet(ruta, columns=cols, engine="pyarrow"))
cobertura = pd.concat(partes, ignore_index=True)
cobertura["periodo"] = pd.to_datetime(cobertura["periodo"], errors="coerce").dt.to_period("M").dt.to_timestamp()
del partes

por_mes = (
    cobertura.groupby("periodo")
    .agg(clientes=("NIU", "nunique"), consumo_kwh=("consumo_kwh_raw", "sum"))
    .sort_index()
)
por_mes["clientes_vs_mes_anterior_pct"] = (por_mes["clientes"].pct_change() * 100).round(1)

meses_todos = pd.date_range(por_mes.index.min(), por_mes.index.max(), freq="MS")
faltantes = [m for m in meses_todos if m not in por_mes.index]

print("HISTÓRICO COMPLETO")
print("-" * 70)
print(f"Archivos anuales : {', '.join(a.name for a in archivos_anuales)}")
print(f"Periodo          : {por_mes.index.min():%Y-%m} -> {por_mes.index.max():%Y-%m}  ({len(por_mes)} meses)")
print(f"NIU distintos    : {cobertura['NIU'].nunique():,}")
if faltantes:
    print("⚠ Meses SIN archivo dentro del rango:", ", ".join(f"{m:%Y-%m}" for m in faltantes))
else:
    print("Sin huecos: todos los meses del rango tienen archivo.")

print("\nÚltimos 15 meses (clientes con fila y consumo crudo; el último mes suele venir incompleto\n"
      "en rurales, eso lo resuelve el borde provisional en los pasos siguientes):")
from IPython.display import display
display(por_mes.tail(15).assign(consumo_kwh=lambda d: d["consumo_kwh"].round(0)))
del cobertura
